In [2]:
import os
import sys
import yaml
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

import multiprocessing as mp
import time
import numpy as np
import itertools
import tools21cm as t2c
from scipy import stats
from tqdm import tqdm
import astropy.units as u
import astropy.constants as cst

plt.rcParams["font.family"] = "serif"

from astropy.io import fits
import casacore.tables as ct

2024-05-29 14:09:53.450063: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## __Analyzing SKA .MS data.__

### Note: The file has only one frequency!

In [ ]:
path= '/scratch/snx3000/vrajkovi/'
ms_file = path + 'lc_256_train_130923_i0_dTgf_ch600_4h1d_256.MS'
ms = ct.table(ms_file, readonly=True, ack=False)

# Get the time and antenna1/antenna2 columns
times = ms.getcol('TIME')
ant1 = ms.getcol('ANTENNA1')
ant2 = ms.getcol('ANTENNA2')

# Get the unique antenna IDs and number of antennas
antennas = np.unique(np.concatenate((ant1, ant2)))
del ant1 , ant2

In [26]:
print(ms)

Table: /scratch/snx3000/vrajkovi/lc_256_train_130923_i0_dTgf_ch600_4h1d_256.MS
188375040 rows
22 columns: UVW FLAG FLAG_CATEGORY WEIGHT SIGMA ANTENNA1 ANTENNA2 ARRAY_ID DATA_DESC_ID EXPOSURE FEED1 FEED2 FIELD_ID FLAG_ROW INTERVAL OBSERVATION_ID PROCESSOR_ID SCAN_NUMBER STATE_ID TIME TIME_CENTROID DATA


In [38]:
spw_table = ct.table(ms.getkeyword('SPECTRAL_WINDOW'), readonly=True, ack=False)
print(spw_table)
freq = spw_table.getcol('CHAN_FREQ')
print(freq)

Table: /scratch/snx3000/vrajkovi/lc_256_train_130923_i0_dTgf_ch600_4h1d_256.MS/SPECTRAL_WINDOW
1 rows
14 columns: MEAS_FREQ_REF CHAN_FREQ REF_FREQUENCY CHAN_WIDTH EFFECTIVE_BW RESOLUTION FLAG_ROW FREQ_GROUP FREQ_GROUP_NAME IF_CONV_CHAIN NAME NET_SIDEBAND NUM_CHAN TOTAL_BANDWIDTH
[[1.66e+08]]


In [37]:
#Storing the antenna layout (x and y) coordinates
ant_table = ct.table(ms.getkeyword('ANTENNA'), readonly=True, ack=False)
print(ant_table)
ant_coords = ant_table.getcol('POSITION')
#np.save('SKA_material/SKA_ant_coords.npy', ant_coords[:,0:-1])

Table: /scratch/snx3000/vrajkovi/lc_256_train_130923_i0_dTgf_ch600_4h1d_256.MS/ANTENNA
512 rows
8 columns: OFFSET POSITION TYPE DISH_DIAMETER FLAG_ROW MOUNT NAME STATION


## __Analyzing LOFAR data to extract the frequencies.__

In [12]:
path= '/scratch/snx3000/vrajkovi/LoTSS_DR2/L242820_SB230_uv.dppp.pre-cal_125145DD7t_166MHz.pre-cal.ms.archive'
ms_file = path
ms = ct.table(ms_file, readonly=True, ack=False)

In [13]:
print(ms)

Table: /scratch/snx3000/vrajkovi/LoTSS_DR2/L242820_SB230_uv.dppp.pre-cal_125145DD7t_166MHz.pre-cal.ms.archive
6798145 rows
25 columns: UVW FLAG_CATEGORY WEIGHT SIGMA ANTENNA1 ANTENNA2 ARRAY_ID DATA_DESC_ID EXPOSURE FEED1 FEED2 FIELD_ID FLAG_ROW INTERVAL OBSERVATION_ID PROCESSOR_ID SCAN_NUMBER STATE_ID TIME TIME_CENTROID DATA FLAG LOFAR_FULL_RES_FLAG WEIGHT_SPECTRUM IMAGING_WEIGHT


In [11]:
spw_table = ct.table(ms.getkeyword('SPECTRAL_WINDOW'), readonly=True, ack=False)
print(spw_table)

Table: /scratch/snx3000/vrajkovi/LoTSS_DR2/L242820_SB230_uv.dppp.pre-cal_125145DD7t_166MHz.pre-cal.ms.archive/SPECTRAL_WINDOW
1 rows
14 columns: MEAS_FREQ_REF REF_FREQUENCY FLAG_ROW FREQ_GROUP FREQ_GROUP_NAME IF_CONV_CHAIN NAME NET_SIDEBAND NUM_CHAN TOTAL_BANDWIDTH CHAN_FREQ CHAN_WIDTH EFFECTIVE_BW RESOLUTION


In [37]:
freq_list = spw_table.getcol('CHAN_FREQ')
num_freqs = len(freq_list[0])

print(freq_list.shape)
print(len(freq_list[0]))

#np.save('LOFAR_material/LOFAR_frequencies.npy', freq_list[0])

(1, 20)
20


In [39]:
uvw_data = ms.getcol('UVW')
vis_data = ms.getcol('DATA')
times = ms.getcol('TIME')
num_times = len(np.unique(times))

ant1 = ms.getcol('ANTENNA1')
ant2 = ms.getcol('ANTENNA2')
num_antennas = len(np.unique(np.concatenate((ant1, ant2))))
num_baselines = int(num_antennas * (num_antennas - 1) / 2)

print(uvw_data.shape)
print(vis_data.shape)
print(times.shape)
print(num_times)
print(num_baselines)

(6798145, 3)
(6798145, 20, 4)
(6798145,)
3595
1891


In [50]:
uvw_data_reshape = uvw_data.reshape(num_times, num_baselines, 3)
vis_data_reshape = vis_data.reshape(num_times, num_baselines, num_freqs, 4)

#np.save('LOFAR_material/LOFAR_baselines.npy', uvw_data_reshape[:,:,0:2])
#np.save('LOFAR_material/LOFAR_visibilities.npy', vis_data_reshape[:,:,:,0])